# Face Recognition System

- There are many ways to do the Face Recognition System

### Ways for Matching I/P image 

- to check whether the input image is matched or not, we can use
- Directly using CNN From Deep Learning Algorithm
- Using ML Model (But need to handle the Images) our case

- For Training the Model also , we use
- CNN or Training the ML Model

### Detecting or taking the Face from i/p -> Face Detection

- We can use YoloV8 which is the best one to detect the face
- Haar Cascades or HOG
- MTCNN

# Using ML Model with FaceNet to create unique numerical representations (embeddings) of faces. and YOLO-V8 for the Face Detection

## Our case Proccess

- Use the OpenCV to extract the VideoCapture
- Then use the Yolo to detect the Face in the photo to get the faces dimensions
- Then save those cropped faces from the images
- use the FaceNet to make the cropped-faces to Embeddings
- Then using SVC to do similarity search and doing face recognition

## Another way

- Same like Our Process
- Only difference is instead of using the YOLO we can use MTCNN
- Face Detection: Use a tool like MTCNN to find the face in an image.
- https://www.kaggle.com/code/yhuan95/face-recognition-with-facenet

## Use of FaceNet

- FaceNet is a highly influential facial recognition model developed by researchers at Google
- It fundamentally changed how face recognition was approached by moving away from traditional classification (identifying specific people) to metric learning (measuring how similar two faces are).
#### Traditional Face-Recognition
- Earlier models that tried to classify a face as "Person A" or "Person B,
- Means when a image(cropped) is send then it tells whethere it is Person A, Person B
#### FaceNet
- Here this model create the Face Embeddings of the image
- FaceNet maps a face image into a 128-dimensional compact Euclidean space
- Every face is converted into a string of 128 numbers, called an embedding.
- In this 128D space, images of the same person are placed very close to each other, while images of different people are pushed far apart
- Once you have these embeddings, you can perform tasks like recognition, verification, and clustering using simple distance formulas (like Squared L2 distance)
- After creating those embeddings then we use SVM Ml Model 

## Importing the Libraries

In [1]:
import sys
print(sys.executable)

C:\Users\heman\anaconda3\envs\project\python.exe


In [2]:
from ultralytics import YOLO

In [3]:
import cv2

In [4]:
from ultralytics import YOLO

In [5]:
import numpy as np

## Taking the photo using the camera

In [8]:
cam  = cv2.VideoCapture(0)
print("Press 'Space' to take a photo, or 'Esc' to exit.")
image_count = 0

Press 'Space' to take a photo, or 'Esc' to exit.


- Initialized the camera . "0" in VideoCapture says the in-build camera

In [9]:
while True:
    # collecting Frame-by-Frame
    ret, frame = cam.read()
    if not ret:
        print("Failed to grab frame")
        break
    cv2.imshow("frame", frame)
    # wait till the key is pressed
    key = cv2.waitKey(1)
    if key % 256 == 27:
        # ESC pressed - Close the program
        print("Closing..")
        break
    elif key % 256 == 32:
        # SPACE pressed - Save the photo
        img_name = f"input_photo{image_count}.png"
        cv2.imwrite(img_name, frame)
        print(f"Photo saved as {img_name}")
        image_count = image_count+1
cam.release()
cv2.destroyAllWindows()

Photo saved as input_photo0.png
Photo saved as input_photo1.png
Photo saved as input_photo2.png
Photo saved as input_photo3.png
Photo saved as input_photo4.png
Closing..


- Making use of Space and esc
- cv2.VideoCapture(0) => capture image from the in-build camera
- cam.read() => returns 1. True/False - says whether the camera is active or not . 2. gives the frame size(it is of array)
- cv2.imshow("Window Name", frame) => Displays the image in a window
- cv2.imwrite("filename.jpg", frame) => Saves the image to your computer.
- cam.release(): It turns off the camera hardware so other apps can use it.

## YOLOv8 -> Face Detection

In [6]:
yolo_model = YOLO('yolov8n-face.pt')
# using the PreTrained model "yolov8n-face.pt"

- yolov8n-face.pt this is the pretrained model file name of what Pretrained model we are using
- In the Project folder we ned to contain a file named yolov8n-face.pt which is downloaded from below Github link 
- Source => https://github.com/akanametov/yolo-face?tab=readme-ov-file
- .pt => for pytorch
- .onnx => for other like c++...

In [19]:
cap = cv2.VideoCapture(0)

print("Starting Face Detection... Press 'ESC' to quit/stop.")
count = 4;

Starting Face Detection... Press 'ESC' to quit/stop.


In [20]:
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Failed to grab frame.")
        break

    # Run YOLOv8 Face Detection
    results = yolo_model(frame, stream=True, conf=0.5, verbose=False)

    detected_face = None

    # gives the multiple faces as results , in that we traverse to each box (each face) to get the co-ordinates of the box to crop
    for r in results:
        boxes = r.boxes
        for box in boxes:
            # Get coordinates (x1, y1, x2, y2) of the box/face detected
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            # Get confidence score
            conf = box.conf[0]
            
            # Draw the bounding box using cv2
            # frame, top-left corner , bottom-right corner, RGB colour , thickness of the rectangular-box
            cv2.rectangle(frame, (x1-1, y1-1), (x2+1, y2+1), (0, 255, 0), 2)

            # Add label with confidence
            confidence_score = f"Face: {conf:.2f}"
            # frame, label text to show, y1-10 -> 10 units margin above the rectangular box,  font-style, font-size, RGB colour, thickness of the text strokes
            cv2.putText(frame, confidence_score, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 0.7, (255,0,0))


            # storing the cropped faces
            detected_face = frame[y1-1:y2+1, x1-1:x2+1]

    # Display the output -> frame
    cv2.imshow("YOLOv8 Face Detection", frame)

    # Exit on 'ESC' key , 's' to save the faces detected
    key = cv2.waitKey(1)
    if key % 256 == ord('s'):
        if detected_face is not None:
            cv2.imwrite(f"img_crop_{count}.jpg", detected_face)
            print(f"Image saved: img_crop_{count}.jpg")
            count += 1
            
    if key % 256 == 27:
        print("Frame closed")
        break

cap.release()
cv2.destroyAllWindows()

Image saved: img_crop_4.jpg
Image saved: img_crop_5.jpg
Frame closed


- Press 's'to take the photo , esc to stop the face detection
- YOLO(): This is the constructor from the Ultralytics library. It loads the neural network weights into memory
 Which means when the pre-trained model is put in the YOLO() constructor then it loads the weights(all parameters) of that model , so that   we no need to train again => as we did for GANs
- 'yolov8n-face.pt': The n stands for Nano, the smallest and fastest version of YOLOv8, optimized for real-time use on CPUs.
- frame: The raw image array from your camera => takes the photo section 
- verbose=False: This disables the text printouts (like "0: 480x640 1 face...") in your terminal, making the console cleaner.   It says how many faces detected,the resolution of our camera, the size of the frames etc..
- conf=0.5: The Confidence Threshold. The model only reports detections if it is at least 50% sure it's a face. But to save the image and check face_recognition we use confidence_score atleast 0.80(80% sure it is a face)

- r.boxes: Contains all detected objects in the current frame.
- box.xyxy[0]: "XYXY" format stands for [xmin, ymin, xmax, ymax].
    xmin, ymin: The top-left corner of the face.
    xmax, ymax: The bottom-right corner of the face.
  Using this we create a rectangular cropped box on the frame using cv2
- box.conf[0]: The exact confidence score (e.g., 0.92) for that specific face.

## Converting the Cropped Faces to Embeddings

In [8]:
from facenet_pytorch import InceptionResnetV1
# feature extraction using the pre-trained InceptionResNetV1 model itself
# InceptionResnetV1 is the FaceNet architecture.

In [9]:
# Load the Pretrained FaceNet Model
facenet_model = InceptionResnetV1(pretrained='vggface2').eval()

- InceptionResnetV1 is the Architecture in FaceNet Model
- vggface2 is the pretrained model in that
- When we loaded that Model for first time then that pretrained model weights,parameters are downloaded
- FaceNet takes RGB images , so we can directly send it ..

In [20]:
facenet_model

InceptionResnetV1(
  (conv2d_1a): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2a): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_2b): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (maxpool_3a): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2d_3b): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU()
  )
  (conv2d_4a): 

In [10]:
import torch
from torchvision import transforms

Embeddings = []

def Create_Embeddings(img_name):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((160,160))
    ])

    img = cv2.imread(f"{img_name}")
    # takes the image
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # convert color to cv2.COLOR_BGR2RGB
    img = transform(img).unsqueeze(0)
    # so after converting to RGB sending it to convert to tensor and resize it t 160X160 size
    # actual image tensor shape is [3, 160, 160] => says rgb,size . unsqueeze is added so that it adds the new dimension [1, 3, 160, 160] => 1 represents the batch size
    with torch.no_grad():
        embedding = facenet_model(img)
    embedding = embedding.squeeze().numpy()
    # actually facenet gives the tensor array as o/p -> [[0.12,-0.23 ....]] => so Embeddings array becomes [[[0.12,-0.23 ...]]] so , first converting it to normal numpy array [0.12, -0.23..]
    # torch.Size([512]) , (512,) => after using convertion to numpy
    Embeddings.append(embedding)

- FaceNet is very specific about the input it receives. You cannot just pass a raw OpenCV image.
- it must be resized to 160x160.
- converted to a PyTorch Tensor, and normalized.
- It takes RGB image
- so when downloaded the FaceNet then it automatically downloads the torch, transforms

In [22]:
# with torch.no_grad():
#     embedding = model(img)

# print(embedding.shape)

# o/p => [1,512] size

In [23]:
# print(embedding)

# o/p => returns the array with the embeddings 

In [11]:
Create_Embeddings("hemanth_6.jpg")
Create_Embeddings("hemanth_7.jpg")
Create_Embeddings("hemanth_8.jpg")
Create_Embeddings("hemanth_9.jpg")
Create_Embeddings("hemanth_1.jpg")
Create_Embeddings("hemanth_2.jpg")
Create_Embeddings("hemanth_4.jpg")
Create_Embeddings("hemanth_5.jpg")
Create_Embeddings("hemanth_10.jpg")
Create_Embeddings("person_1.jpg")
Create_Embeddings("person_2.jpg")
Create_Embeddings("person_3.jpg")
Create_Embeddings("person_4.jpg")

In [16]:
# Embeddings = [
#    [0.12, -0.33, ...],   # image 1
#    [0.11, -0.30, ...],   # image 2
# ]
# embeddigs look like this after converting to numpy array

In [12]:
len(Embeddings)

13

In [13]:
X = np.array(Embeddings)

In [14]:
print(X.shape)
print(X)

(13, 512)
[[  0.0023637   -0.044962   -0.016202 ...    0.060312    0.008489   -0.039069]
 [  -0.020036   0.0051533   -0.026319 ...    0.021003    0.090982  -0.0038287]
 [   0.045439   -0.041702   -0.044274 ...    0.019588   -0.013102   -0.012256]
 ...
 [  -0.018952   0.0062302   -0.032143 ...   0.0043616   -0.037882    0.049153]
 [  -0.023053    0.024975  -0.0026751 ...    0.019092   0.0030461    0.041566]
 [   0.018228  -0.0083717   0.0070001 ...   -0.042296   -0.044525    0.018238]]


In [15]:
Y = ["hemanth","hemanth","hemanth","hemanth","hemanth","hemanth","hemanth","hemanth","hemanth","person","person","person","person"]
Y = np.array(Y)

## Training SVM ML Model

In [16]:
from sklearn.svm import SVC

In [17]:
svc_model = SVC(kernel='linear', probability=True)
svc_model.fit(X, Y)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


## Testing

In [18]:
import torch
from torchvision import transforms

def test_Embeddings(img_name):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((160,160))
    ])

    img = cv2.imread(f"{img_name}")
    # takes the image
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # convert color to cv2.COLOR_BGR2RGB
    img = transform(img).unsqueeze(0)
    # so after converting to RGB sending it to convert to tensor and resize it t 160X160 size
    # actual image tensor shape is [3, 160, 160] => says rgb,size . unsqueeze is added so that it adds the new dimension [1, 3, 160, 160] => 1 represents the batch size
    with torch.no_grad():
        embedding = facenet_model(img)
    embedding = embedding.squeeze().numpy()
    # actually facenet gives the tensor array as o/p -> [[0.12,-0.23 ....]] => so Embeddings array becomes [[[0.12,-0.23 ...]]] so , first converting it to normal numpy array [0.12, -0.23..]
    # torch.Size([512]) , (512,) => after using convertion to numpy
    return embedding

In [21]:
test = test_Embeddings("test1.jpg")

In [22]:
test = np.array(test)

In [23]:
len(test)

512

In [24]:
test.shape

(512,)

In [25]:
test = test.reshape(1, -1)

In [26]:
test.shape

(1, 512)

In [27]:
print(test)

[[   0.044048   -0.025155   -0.053119    0.050477    -0.05269   0.0034806   0.0063471   0.0062064  -0.0086668   -0.065319    0.006567   -0.063518   0.0068019 -0.00085772    0.055045     0.01335    0.025726   -0.022084    0.025136    -0.01579    -0.03438   -0.021405    0.060609     0.10815    0.043733    0.092012
    0.0084547   -0.062252   0.0063472    0.016654   -0.095456    0.052019    -0.11107    0.012443   -0.065001    0.086724   -0.042234    0.044194   -0.037444    0.025902    0.011654    0.010697   -0.049079   -0.047455    -0.01066   -0.015354  -0.0099083    0.097069   -0.049266    0.052935    0.031488    0.052645
   -0.0041263    0.025542   -0.067534    0.028986  -0.0036536  -0.0012226    0.080547    0.011209    0.017309   -0.031449    0.034121   -0.015117    0.074476   0.0052118   -0.054957   -0.043343   -0.020236    0.039306   -0.040609    0.057952  0.00073267  -0.0030237    0.026611  -0.0087118   -0.061554   -0.042908
   -0.0081332    0.016274    0.062745   0.0094188   -0.060

In [28]:
prediction = svc_model.predict(test)
confidence = svc_model.predict_proba(test).max()

print(prediction[0], ". "  "confidence score" " - >", confidence)

hemanth . confidence score - > 0.9370388141661725


In [29]:
if confidence>=0.80:
    print("You are Authorized")
else:
    print("Unauthorized")

You are Authorized


#### Test2

In [30]:
test2 = test_Embeddings("Test2.jpg")

In [31]:
test2 = np.array(test2)
test2 = test2.reshape(1, -1)
print(test2.shape)

(1, 512)


In [32]:
print(test2)

[[   0.052611   -0.064325    -0.05156    0.016688   -0.065414   -0.053845    0.028856   0.0012253   -0.017254   -0.030863  -0.0035402   -0.030375   -0.038907  -0.0079847    -0.04042    0.072167   -0.011979     0.02666  -0.0056364  -0.0037275    0.054451   -0.022948    0.028338    0.028668   0.0080869    0.061785
    -0.024784   -0.066907    0.018537   -0.024167   0.0013138    0.013256    -0.02382    0.012708   -0.056019    0.072802  -0.0058118    0.012523   -0.005915    0.038593   0.0027272   -0.045666    -0.03902    0.032672    0.013517   -0.031462   -0.097551    0.062157   -0.054472    0.098477    0.026053   -0.015914
    0.0019606    0.058451   -0.052025   -0.013418   0.0091586   -0.012112    0.018955     -0.0515    0.056466    0.036276   -0.017132    0.048083    0.024107  -0.0054631   0.0035578  -0.0092651  -0.0038506    0.013178   -0.051731    0.011527     0.01327    0.031861   -0.011926  -0.0013053   -0.029488    0.026629
    -0.026776   -0.011323    0.029482   0.0063701   -0.051

In [33]:
prediction = svc_model.predict(test2)
confidence = svc_model.predict_proba(test2).max()

print(prediction[0], ". "  "confidence score" " - >", confidence)

hemanth . confidence score - > 0.9229602474491362


In [34]:
if confidence>=0.70:
    print("You are Authorized")
else:
    print("Unauthorized")

You are Authorized
